# 02. Databricks: The Metric View (Bidirectional Demo)

You visit this notebook three times. Run only the step you are on.

| Visit | Step | What happens |
|---|---|---|
| 1 | Step 1 | Same Iceberg data as Snowflake, and no Metric View yet |
| 2 | Step 2 | Build the Metric View from Ossie, add a measure, export it back |
| 3 | Step 3 | The metric Snowflake added arrives here on its own |

Setup must already have been run: `00_snowflake_setup.sql`, then
`00_databricks_setup.ipynb`.

## Install the Apache Ossie converter

Snowflake reads and writes Ossie natively. Databricks does not yet, so this side uses
the open-source converter from the Apache project, pinned to a known commit.

On serverless this install takes most of a minute, so run these two cells before the
audience is watching. For a tighter demo, install `ossie-databricks` as a cluster library
and skip them.

In [0]:
%pip install "git+https://github.com/apache/ossie.git@01058aa416423cf43a74e7f9fb7f5f70981a418e#subdirectory=converters/databricks"

In [0]:
dbutils.library.restartPython()

## Configuration

In [0]:
CATALOG       = "demos"
SCHEMA        = "ext_semantic_interop"
METRIC_VIEW   = f"{CATALOG}.{SCHEMA}.sales_metric_view"
MODEL_NAME    = "SALES_SV"          # keep Snowflake's model name across the round trip

# Ossie carries Snowflake's 3-part table names; swap namespaces on the way through.
SF_NAMESPACE  = "DEMOS.EXT_SEMANTIC_INTEROP"
DBX_NAMESPACE = f"{CATALOG}.{SCHEMA}"

S3_BUCKET     = "s3://snowflake-ossie-interop"                  # <-- your bucket
OSSIE_MODEL   = f"{S3_BUCKET}/ossie/sales_model.yaml"           # shared with Snowflake
STATE_FILE    = f"{S3_BUCKET}/ossie/_state/databricks.json"     # this side's bookmark

print(f"Metric View  : {METRIC_VIEW}")
print(f"Shared model : {OSSIE_MODEL}")

---

# Step 2 - Build the Metric View from Ossie

Run this on your **second** visit, after Snowflake has written `sales_model.yaml`.

## 2a. Why Databricks needs a small shim

The Ossie spec is moving quickly, which is what you want from a young open standard.
Right now Snowflake emits spec `0.1.1` and the Apache Databricks converter targets
`0.2.0.dev0`, and the two disagree on three mechanical points: the version string, the
dialect label on expressions, and where metrics live in the document.

The shim below reconciles exactly those three things. It touches the envelope, never the
semantics: no name, expression or relationship is altered. As the spec settles this goes
away.

Worth saying plainly: Snowflake having native Ossie support in both directions, and the
converter being open source and vendor-neutral, is what makes this a 40-line adapter
rather than an integration project.

## 2b. The two functions

One cell, two directions. `import_ossie_to_metric_view` reads Ossie and returns Metric
View YAML; `export_metric_view_to_ossie` does the reverse. Everything below is plumbing
around the converter.

In [0]:
import json
import re

import yaml
from ossie_databricks import convert_metric_view_to_ossie, convert_ossie_to_metric_view

CONVERTER_VERSION = "0.2.0.dev0"     # what the Apache converter requires
SNOWFLAKE_VERSION = "0.1.1"          # what Snowflake emits and expects
SNOWFLAKE, ANSI, DATABRICKS = "SNOWFLAKE", "ANSI_SQL", "DATABRICKS"


def _relabel(expression, frm, to):
    """Swap the dialect label on an Ossie expression, leaving the expression alone."""
    for dialect in (expression or {}).get("dialects", []) or []:
        if dialect.get("dialect") == frm:
            dialect["dialect"] = to


def shim_snowflake_to_converter(ossie_yaml):
    """Snowflake Ossie 0.1.1 -> converter-ready 0.2.0.dev0."""
    root = yaml.safe_load(ossie_yaml)
    root["version"] = CONVERTER_VERSION

    for model in root.get("semantic_model") or []:
        hoisted = []
        for ds in model.get("datasets") or []:
            unqualify = re.compile(re.escape(ds.get("name", "")) + r"\.", re.IGNORECASE)

            # Snowflake hides metrics in a dataset extension; the converter wants them
            # at model level with bare column names.
            keep = []
            for ext in ds.get("custom_extensions") or []:
                if ext.get("vendor_name") != SNOWFLAKE:
                    keep.append(ext)
                    continue
                for metric in json.loads(ext.get("data") or "{}").get("metrics") or []:
                    hoisted.append({
                        "name": metric["name"],
                        "expression": {"dialects": [
                            {"dialect": ANSI, "expression": unqualify.sub("", metric["expr"])}]},
                    })
            ds["custom_extensions"] = keep
            if not keep:
                ds.pop("custom_extensions")

            # Facts are Snowflake-only; dropping them keeps them out of the Metric View
            # dimension list. They live on inside the measure expressions.
            fields = []
            for field in ds.get("fields") or []:
                _relabel(field.get("expression"), SNOWFLAKE, ANSI)
                field.pop("custom_extensions", None)
                if "dimension" in field:
                    fields.append(field)
            ds["fields"] = fields
            if not fields:
                ds.pop("fields")

        model["metrics"] = (model.get("metrics") or []) + hoisted

        # Metrics already at model level need the same relabelling, or the converter
        # finds no readable dialect and silently drops every measure.
        names = [ds.get("name", "") for ds in model.get("datasets") or []]
        for metric in model["metrics"]:
            _relabel(metric.get("expression"), SNOWFLAKE, ANSI)
            for dialect in (metric.get("expression") or {}).get("dialects") or []:
                for name in names:
                    dialect["expression"] = re.sub(
                        re.escape(name) + r"\.", "", dialect.get("expression", ""),
                        flags=re.IGNORECASE)

    return yaml.safe_dump(json.loads(json.dumps(root)), sort_keys=False)


def shim_converter_to_snowflake(ossie_yaml, model_name=None):
    """Converter Ossie -> Snowflake-importable 0.1.1."""
    root = yaml.safe_load(ossie_yaml)
    root["version"] = SNOWFLAKE_VERSION

    for model in root.get("semantic_model") or []:
        datasets = model.get("datasets") or []
        for ds in datasets:
            ds["name"] = ds["name"].upper()
            if ds.get("unique_keys"):            # Snowflake wants primary_key back
                ds["primary_key"] = ds.pop("unique_keys")[0]
        for rel in model.get("relationships") or []:
            rel["from"], rel["to"] = rel["from"].upper(), rel["to"].upper()

        fact = datasets[0]["name"] if datasets else None
        qualify = lambda e: re.sub(
            r"(?<![\w.])([A-Za-z_]\w*)(?!\s*\()(?![\w.])", lambda m: f"{fact}.{m.group(1)}", e)

        for ds in datasets:
            for field in ds.get("fields") or []:
                _relabel(field.get("expression"), DATABRICKS, SNOWFLAKE)
                if ds["name"] != fact:
                    field.setdefault("dimension", {})

        # Snowflake requires the fact columns a metric references to exist as fields.
        referenced, pattern = [], re.compile(re.escape(fact or "") + r"\.([A-Za-z_]\w*)")
        for metric in model.get("metrics") or []:
            _relabel(metric.get("expression"), DATABRICKS, SNOWFLAKE)
            for dialect in (metric.get("expression") or {}).get("dialects") or []:
                dialect["expression"] = qualify(dialect["expression"])
                referenced += [c for c in pattern.findall(dialect["expression"])]
        if fact:
            existing = {f["name"].lower() for f in datasets[0].get("fields") or []}
            for column in dict.fromkeys(referenced):
                if column.lower() not in existing:
                    datasets[0].setdefault("fields", []).append({
                        "name": column.upper(),
                        "expression": {"dialects": [
                            {"dialect": SNOWFLAKE, "expression": column}]}})
        if model_name:
            model["name"] = model_name

    return yaml.safe_dump(json.loads(json.dumps(root)), sort_keys=False)


# ------------------------------------------------------------ the two directions

def import_ossie_to_metric_view(path=None):
    """Read Ossie from S3 and replace the Metric View with it. Returns the measures."""
    ossie = dbutils.fs.head(path or OSSIE_MODEL, 1024 * 1024)
    mv_yaml = convert_ossie_to_metric_view(shim_snowflake_to_converter(ossie))
    mv_yaml = mv_yaml.replace(SF_NAMESPACE, DBX_NAMESPACE)

    mv = yaml.safe_load(mv_yaml)
    for join in mv.get("joins") or []:
        join.pop("rely", None)               # not accepted by older serdes
    mv_yaml = yaml.safe_dump(mv, sort_keys=False)

    spark.sql(f"CREATE OR REPLACE VIEW {METRIC_VIEW} "
              f"WITH METRICS LANGUAGE YAML AS $$\n{mv_yaml}\n$$")
    return [m["name"] for m in mv.get("measures") or []]


def metric_view_as_ossie():
    """The deployed Metric View, as Snowflake-importable Ossie. Writes nothing."""
    ddl = spark.sql(f"SHOW CREATE TABLE {METRIC_VIEW}").collect()[0][0]
    start = ddl.index("$") + 2
    mv_yaml = ddl[start:ddl.index("$", start)].strip()

    ossie = convert_metric_view_to_ossie(mv_yaml).replace(DBX_NAMESPACE, SF_NAMESPACE)
    return shim_converter_to_snowflake(ossie, model_name=MODEL_NAME)


def export_metric_view_to_ossie():
    """Publish the Metric View to the shared Ossie file on S3."""
    ossie = metric_view_as_ossie()
    dbutils.fs.put(OSSIE_MODEL, ossie, overwrite=True)
    return ossie


print("Ready: import_ossie_to_metric_view() and export_metric_view_to_ossie()")

## 2c. Build the Metric View

Everything above was setup. This is the actual step: read the file Snowflake wrote, and
the Metric View exists.

In [0]:
# measures = import_ossie_to_metric_view()

# print(f"Created {METRIC_VIEW}")
# print("Measures:", ", ".join(measures))

## 2g. Keep picking up future changes automatically

Manual is fine for a first pass. From here on we want this side to notice Snowflake-side
changes on its own.

`sync_once()` below does that. It also uses a small consistency function so that two
platforms polling the same file settle instead of overwriting each other in a loop. The
mechanics are in `assets/ossie_sync/` if anyone asks; they are not the interesting part
of this demo.

In [0]:
# Consistency helpers, generated from assets/ossie_sync/ by assets/build_notebooks.py.
import hashlib
from datetime import datetime, timezone

# --- BEGIN GENERATED: ossie_sync.fingerprint ---
# Generated from assets/ossie_sync/fingerprint.py by assets/build_notebooks.py.
# Edit that file and re-run the build; changes made here are overwritten.
"""Canonical semantic fingerprint for an Ossie document.

Why this exists
---------------
The Snowflake <-> Databricks round trip is not byte-stable. The same semantic model
comes back with different dataset name casing, `primary_key` renamed to `unique_keys`,
facts dropped, dialect labels rewritten, and expressions gaining or losing table
qualifiers (`SUM(orders.order_amount)` on one side, `SUM(order_amount)` on the other).

So a sync that compares raw YAML, or a hash of it, never sees the two sides as equal and
writes forever. A sync that compares file timestamps is worse: every write makes the
writer the most recent change, so the model ping-pongs between platforms.

`semantic_fingerprint` solves this by hashing only the part of the model that both
platforms can express, in a normalized form that survives the trip. Two models with the
same fingerprint are treated as the same model, which is what lets the sync go quiet.

What is included
----------------
    tables          alias and source table, lowercased, last path component only
    relationships   from, to, and the join columns
    dimensions      qualified dimension names, sorted
    metrics         name and expression, sorted, table qualifiers stripped

What is excluded, and why
-------------------------
    Ossie `version`         differs by spec revision (0.1.1 against 0.2.0.dev0)
    dialect labels          SNOWFLAKE against ANSI_SQL against DATABRICKS
    comments, descriptions  Databricks does not round-trip them
    FACTS                   Snowflake-only concept, dropped by the converter
    relationship names      the converter rewrites their casing
    primary keys            Snowflake `primary_key` becomes `unique_keys` and back
    field and key order     not semantically meaningful

Excluding these has a real cost: editing only a comment, or only a fact, propagates
nothing. That is the deliberate trade. Including them would mean the two sides never
agree and the sync would write on every tick forever.
"""



FINGERPRINT_VERSION = "1"

# Matches a leading table qualifier on a column reference, e.g. the "orders." in
# "orders.order_amount". Stripped so that SUM(orders.order_amount) on the Snowflake side
# and SUM(order_amount) on the Databricks side produce the same fingerprint.
_QUALIFIER = re.compile(r"\b[A-Za-z_]\w*\.(?=[A-Za-z_]\w*)")
_WHITESPACE = re.compile(r"\s+")


def normalize_expression(expr):
    """Reduce a SQL expression to a comparable form.

    Lowercases, collapses whitespace, strips table qualifiers, and removes spaces
    around punctuation so that formatting differences do not register as changes.

        >>> normalize_expression("SUM( orders.order_amount )")
        'sum(order_amount)'
        >>> normalize_expression("sum(order_amount)")
        'sum(order_amount)'
    """
    if not expr:
        return ""
    text = _QUALIFIER.sub("", str(expr))
    text = _WHITESPACE.sub(" ", text).strip().lower()
    for token in ("(", ")", ",", "+", "-", "*", "/"):
        text = text.replace(" " + token, token).replace(token + " ", token)
    return text


def _last_identifier(source):
    """DEMOS.EXT_SEMANTIC_INTEROP.ORDERS -> orders"""
    return str(source or "").split(".")[-1].strip().strip('"').lower()


def _pick_expression(expression_obj):
    """Return the first expression string from an Ossie expression object.

    Dialect is ignored on purpose. The same expression labelled SNOWFLAKE, ANSI_SQL or
    DATABRICKS is the same expression for fingerprint purposes.
    """
    if not isinstance(expression_obj, dict):
        return ""
    for dialect in expression_obj.get("dialects") or []:
        if dialect.get("expression"):
            return dialect["expression"]
    return ""


def semantic_projection(ossie_yaml):
    """Reduce an Ossie document to the platform-neutral structure that gets hashed.

    Returned separately from the hash so notebooks can print it and show exactly what
    is being compared. When a sync will not converge, diffing two projections is the
    fastest way to find out which field is to blame.
    """
    root = yaml.safe_load(ossie_yaml) if isinstance(ossie_yaml, str) else ossie_yaml
    models = root.get("semantic_model") or []

    tables, relationships, dimensions, metrics = [], [], [], []

    for model in models:
        datasets = model.get("datasets") or []

        for dataset in datasets:
            alias = _last_identifier(dataset.get("name"))
            tables.append({"alias": alias, "source": _last_identifier(dataset.get("source"))})

            for field in dataset.get("fields") or []:
                # Only dimensions are portable. Snowflake facts have no `dimension` key
                # and are dropped by the Databricks converter, so including them here
                # would break convergence.
                if "dimension" not in field:
                    continue
                dimensions.append(f"{alias}.{str(field.get('name','')).lower()}")

        for rel in model.get("relationships") or []:
            # Ossie spells the join columns from_columns / to_columns. Only the join
            # columns are fingerprinted; the relationship's own name is not, because the
            # converter rewrites its casing (ORDERS_TO_CUSTOMERS -> ORDERS_to_CUSTOMERS).
            relationships.append({
                "from": _last_identifier(rel.get("from")),
                "to": _last_identifier(rel.get("to")),
                "from_columns": sorted(_last_identifier(c) for c in rel.get("from_columns") or []),
                "to_columns": sorted(_last_identifier(c) for c in rel.get("to_columns") or []),
            })

        # Snowflake stores metrics inside datasets[*].custom_extensions as a JSON blob;
        # the Apache converter uses a top-level `metrics` list. Read both.
        for metric in model.get("metrics") or []:
            metrics.append({
                "name": str(metric.get("name", "")).lower(),
                "expr": normalize_expression(_pick_expression(metric.get("expression"))),
            })

        for dataset in datasets:
            for ext in dataset.get("custom_extensions") or []:
                if ext.get("vendor_name") != "SNOWFLAKE":
                    continue
                try:
                    blob = json.loads(ext.get("data") or "{}")
                except (ValueError, TypeError):
                    continue
                for metric in blob.get("metrics") or []:
                    metrics.append({
                        "name": str(metric.get("name", "")).lower(),
                        "expr": normalize_expression(metric.get("expr")),
                    })

    def dedupe(rows, key):
        seen, out = set(), []
        for row in rows:
            marker = key(row)
            if marker not in seen:
                seen.add(marker)
                out.append(row)
        return out

    return {
        "fingerprint_version": FINGERPRINT_VERSION,
        "tables": sorted(dedupe(tables, lambda t: t["alias"]), key=lambda t: t["alias"]),
        "relationships": sorted(
            dedupe(relationships, lambda r: (r["from"], r["to"], tuple(r["from_columns"]))),
            key=lambda r: (r["from"], r["to"]),
        ),
        "dimensions": sorted(set(dimensions)),
        "metrics": sorted(dedupe(metrics, lambda m: m["name"]), key=lambda m: m["name"]),
    }


def semantic_fingerprint(ossie_yaml):
    """sha256 over the canonical projection. Stable across the round trip."""
    canonical = json.dumps(semantic_projection(ossie_yaml), sort_keys=True, separators=(",", ":"))
    return "sha256:" + hashlib.sha256(canonical.encode("utf-8")).hexdigest()


def describe(fingerprint):
    """Short form for log lines and notebook output."""
    if not fingerprint:
        return "(none)"
    return fingerprint.split(":")[-1][:12]
# --- END GENERATED: ossie_sync.fingerprint ---

# --- BEGIN GENERATED: ossie_sync.decide ---
# Generated from assets/ossie_sync/decide.py by assets/build_notebooks.py.
# Edit that file and re-run the build; changes made here are overwritten.
"""The sync decision: compare three fingerprints, return one verdict.

Both platforms and both architectures run this same function. The only thing that varies
is `allowed`, which is what stops the unidirectional variant from being a fork of the
bidirectional one.

The three inputs
----------------
    local   fingerprint of the model as it exists on this platform right now
    remote  fingerprint of the shared Ossie file on S3
    base    fingerprint this platform last agreed on, from its own state file

`base` is what makes this terminate. Without it there is no way to tell "the other side
changed" from "I changed", so both sides write and the model ping-pongs forever. With it,
each side can see which of the two moved, act once, record the new base, and go quiet.
"""

NO_CHANGE = "NO_CHANGE"
ADOPT = "ADOPT"
IMPORT = "IMPORT"
EXPORT = "EXPORT"
CONFLICT = "CONFLICT"
REVERT_LOCAL_DRIFT = "REVERT_LOCAL_DRIFT"

BIDIRECTIONAL = ("IMPORT", "EXPORT")
SNOWFLAKE_MANAGED_SOURCE = ("EXPORT",)      # Snowflake in the managed architecture
SNOWFLAKE_MANAGED_MIRROR = ("IMPORT",)      # Databricks in the managed architecture

REASONS = {
    NO_CHANGE: "local and shared model agree, nothing to do",
    ADOPT: "no recorded base, taking the shared model as the starting point",
    IMPORT: "shared model changed, replacing the local model",
    EXPORT: "local model changed, publishing to the shared Ossie file",
    CONFLICT: "both sides changed since the last agreement",
    REVERT_LOCAL_DRIFT: "local edit is not authoritative, restoring from the shared model",
}


class Decision:
    """A verdict plus the fingerprints that produced it, so it can be logged and read."""

    def __init__(self, action, reason, local, remote, base):
        self.action = action
        self.reason = reason
        self.local = local
        self.remote = remote
        self.base = base

    @property
    def writes(self):
        return self.action in (IMPORT, EXPORT, ADOPT, REVERT_LOCAL_DRIFT)

    def __str__(self):
        return f"{self.action} - {self.reason}"

    def to_dict(self):
        return {
            "action": self.action,
            "reason": self.reason,
            "local_fingerprint": self.local,
            "remote_fingerprint": self.remote,
            "base_fingerprint": self.base,
        }


def decide(local, remote, base, allowed=BIDIRECTIONAL, conflict_winner=None, platform=None):
    """Return a Decision.

    allowed
        Which write directions this platform may take. Bidirectional passes both.
        The managed architecture passes ("EXPORT",) on Snowflake and ("IMPORT",) on
        Databricks; an EXPORT that is not allowed becomes REVERT_LOCAL_DRIFT.

    conflict_winner, platform
        When both sides changed, the platform named by `conflict_winner` keeps its
        version. Anything else imports. Demoware: the losing edit is discarded with
        nothing more than a log line. See docs/PRODUCTION_ARCHITECTURE.md.
    """
    def verdict(action):
        return Decision(action, REASONS[action], local, remote, base)

    if local and remote and local == remote:
        return verdict(NO_CHANGE)

    if not local:
        # Nothing here yet, so there is no local change to protect.
        return verdict(ADOPT if remote else NO_CHANGE)

    if not remote:
        # Local model exists but the shared file does not.
        return verdict(EXPORT if EXPORT in allowed else NO_CHANGE)

    if base is None:
        return verdict(ADOPT)

    if local == base:
        action = IMPORT
    elif remote == base:
        action = EXPORT
    else:
        if conflict_winner and platform and conflict_winner == platform:
            return verdict(EXPORT if EXPORT in allowed else CONFLICT)
        if conflict_winner and platform:
            return verdict(IMPORT if IMPORT in allowed else CONFLICT)
        return verdict(CONFLICT)

    if action == EXPORT and EXPORT not in allowed:
        # Managed architecture: a locally edited mirror is drift, not a contribution.
        return verdict(REVERT_LOCAL_DRIFT)
    if action == IMPORT and IMPORT not in allowed:
        return verdict(NO_CHANGE)

    return verdict(action)


def next_base(decision):
    """The fingerprint to record after acting, or None to leave the base unchanged."""
    if decision.action in (IMPORT, ADOPT, REVERT_LOCAL_DRIFT):
        return decision.remote
    if decision.action == EXPORT:
        return decision.local
    return None
# --- END GENERATED: ossie_sync.decide ---

# --- BEGIN GENERATED: ossie_sync.state ---
# Generated from assets/ossie_sync/state.py by assets/build_notebooks.py.
# Edit that file and re-run the build; changes made here are overwritten.
"""Per-platform sync state, stored as JSON next to the shared Ossie file.

    s3://<bucket>/ossie/
      sales_model.yaml            shared model, either side may write
      _state/snowflake.json       written only by Snowflake
      _state/databricks.json      written only by Databricks

One writer per file, so there is no lock and no race. Each side reads only its own state
to answer "what did I last agree to", which is the `base` argument to decide().

Reading and writing the file is left to the caller, because the two runtimes do it very
differently: Databricks has dbutils.fs, Snowflake has stage COPY INTO. These helpers only
handle the JSON shape.
"""


STATE_VERSION = "1"


def new_state(base_fingerprint=None, last_action=None, platform=None):
    return {
        "state_version": STATE_VERSION,
        "base_fingerprint": base_fingerprint,
        "last_action": last_action,
        "by": platform,
        "at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }


def parse_state(text):
    """Tolerant read. A missing, empty or corrupt state file means no recorded base.

    Returning an empty state rather than raising is deliberate: decide() treats a base of
    None as ADOPT, which is the safe first-run behaviour and also the recovery path if
    someone deletes the file mid-demo.
    """
    if not text:
        return new_state()
    try:
        loaded = json.loads(text)
    except (ValueError, TypeError):
        return new_state()
    if not isinstance(loaded, dict):
        return new_state()
    return {
        "state_version": loaded.get("state_version", STATE_VERSION),
        "base_fingerprint": loaded.get("base_fingerprint"),
        "last_action": loaded.get("last_action"),
        "by": loaded.get("by"),
        "at": loaded.get("at"),
    }


def base_of(text):
    """The recorded base fingerprint from raw state-file text, or None."""
    return parse_state(text).get("base_fingerprint")


def serialize_state(state):
    return json.dumps(state, indent=2, sort_keys=True)


def state_after(decision, platform, next_base_fingerprint):
    """Build the state to persist after acting on a decision."""
    return new_state(
        base_fingerprint=next_base_fingerprint or decision.base,
        last_action=decision.action,
        platform=platform,
    )
# --- END GENERATED: ossie_sync.state ---

In [0]:
PLATFORM = "databricks"


def sync_once():
    """Bring this side into agreement with the shared model. Safe to run repeatedly."""
    shared = None
    try:
        shared = dbutils.fs.head(OSSIE_MODEL, 1024 * 1024)
    except Exception:
        pass

    try:
        state = dbutils.fs.head(STATE_FILE, 1024 * 1024)
    except Exception:
        state = None

    local_model = None
    try:
        local_model = metric_view_as_ossie()
    except Exception:
        pass          # no Metric View yet

    local = semantic_fingerprint(local_model) if local_model else None
    remote = semantic_fingerprint(shared) if shared else None
    verdict = decide(local, remote, base_of(state),
                     conflict_winner="snowflake", platform=PLATFORM)

    if verdict.action in (IMPORT, ADOPT, REVERT_LOCAL_DRIFT):
        measures = import_ossie_to_metric_view()
        print(f"{verdict.action}: applied the shared model -> {', '.join(measures)}")
    elif verdict.action == EXPORT:
        export_metric_view_to_ossie()
        print(f"{verdict.action}: published this side's model")
    else:
        print(verdict)

    new_base = next_base(verdict)
    if new_base:
        dbutils.fs.put(STATE_FILE,
                       serialize_state(state_after(verdict, PLATFORM, new_base)),
                       overwrite=True)
    return verdict.action


sync_once()   # already in agreement, so this reports NO_CHANGE

### Schedule it

Workflows -> Jobs -> Create job:

- Task type **Notebook**, pointing at this notebook
- Schedule **every minute**
- **Max concurrent runs 1**, so a slow run never overlaps the next
- Serverless, or a small cluster kept warm

One caveat if you schedule the whole notebook: it would run every cell. For a real
schedule, put `sync_once()` in its own small notebook, or wrap the demo cells in an
`if False:`. For the demo it is enough to know the function is the unit of automation and
to trigger it by hand.

Pause the job when the demo ends.